# Sports Ticket Classification with Logistic Regression – Solution

**Short name (GitHub):** `Bet_LogReg`  
Worked key for the sportsbook adaptation of `Income_LogReg`. Numbers from scikit-learn on `data/sports_tickets.csv` (12,000 rows), `random_state=1`, `test_size=0.2`.

**Not betting advice.** Synthetic blotter. A 0.63 AUC does not cover −110 juice.

| Metric | Value |
|--------|-------|
| Class mix | 61.08% Lost / 38.92% Won |
| X after dummies | 12,000 × 12 |
| Intercept | ≈ +0.21 |
| Test accuracy | ≈ 0.627 |
| Test confusion | TN 1305, FP 146, FN 749, TP 200 |
| Won precision / recall / F1 | 0.58 / 0.21 / 0.31 |
| ROC AUC | ≈ 0.633 |
| L1 zeros | 3 of 12 |

## Inline cheat-sheet

Same table as the skeleton — see **`Bet_LogReg_Cheatsheet.docx`**. Signs to recover: `clv_cents` +, `line_move` +, `public_pct` −, and on the expanded card `injury_star` −.

## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load

In [ ]:
df = pd.read_csv("data/sports_tickets.csv")
print(df.head())
print(df.shape)
print(df.dtypes)

## 2. Imbalance

Always-predict-Lost baseline ≈ 0.611. Lesson accuracy 0.627 is only about **1.6 points** of lift. Lead with AUC and Won-recall.

In [ ]:
print(df.result.value_counts())
print(df.result.value_counts(normalize=True).round(4))

## 2.2 Dummy-encode

In [ ]:
feature_cols = [
    "clv_cents", "public_pct", "line_move", "hours_to_start",
    "stake_usd", "is_home", "sport", "market",
]
X = pd.get_dummies(df[feature_cols], drop_first=True).astype(float)
print(X.shape)
print(list(X.columns))

## 2.3 Heatmap

Sport dummies anti-correlate (a ticket is one sport). Market dummies the same. No emergency drop.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

## 2.4 Scale diagnosis + y

`stake_usd` runs 5–2,000; `public_pct` runs 18–92; `clv_cents` is roughly −50 to +50. Unscaled L1 still fits; scale when you want per-SD story for the desk.

In [ ]:
for c in ["clv_cents", "public_pct", "line_move", "hours_to_start", "stake_usd"]:
    print(f"{c:16s} min={X[c].min():8.2f}  max={X[c].max():8.2f}  mean={X[c].mean():8.2f}")
y = np.where(df.result == "Lost", 0, 1)
print("cash rate", round(y.mean(), 4))

## 3. Fit

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, random_state=1, test_size=0.2
)
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
print(x_train.shape, x_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))

### Parameters

Intercept ≈ +0.21. Largest positive working features: `line_move`, `clv_cents`, NHL/NBA relative to MLB. `public_pct` is negative (fading the public).

In [ ]:
print("Model Parameters, Intercept:")
print(log_reg.intercept_[0])
print("Model Parameters, Coeff:")
print(log_reg.coef_)

### Confusion + accuracy

[[1305, 146], [749, 200]]. Accuracy ≈ 0.627. At t = 0.5 the model posts very few “Won” calls (346 of 2,400) — a tight hold list.

In [ ]:
print("Confusion Matrix on test set:")
print(confusion_matrix(y_test, y_pred))
print("Accuracy Score on test set:")
print(log_reg.score(x_test, y_test))
print(classification_report(y_test, y_pred, digits=3))

## 4. Coef table + bar

In [ ]:
coef_df = (
    pd.DataFrame({"var": x_train.columns, "coef": log_reg.coef_[0]})
    .query("coef.abs() > 0")
    .sort_values("coef")
)
print(coef_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x="var", y="coef", color="#2c7bb6")
plt.xticks(rotation=90)
plt.title("LR Coefficient Values (ticket won)")
plt.tight_layout()
plt.show()

## 5. ROC

AUC ≈ 0.633. Ranking is better than chance; it is not a close-beating machine after juice.

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC score:", roc_auc)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure()
plt.plot(fpr, tpr, color="darkorange", label="ROC curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — ticket cashed")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.show()

## 6. Alternates

### Scaled L1 — CLV becomes readable per SD

In [ ]:
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(C=0.05, penalty="l1", solver="liblinear")),
])
pipe.fit(x_train, y_train)
p = pipe.predict_proba(x_test)[:, 1]
print("scaled acc", pipe.score(x_test, y_test))
print("scaled auc", roc_auc_score(y_test, p))
print(pd.DataFrame({
    "var": x_train.columns,
    "scaled_coef": pipe.named_steps["lr"].coef_[0],
}).sort_values("scaled_coef").to_string(index=False))

### L2 — no exact zeros

In [ ]:
log_l2 = LogisticRegression(max_iter=2000)
log_l2.fit(x_train, y_train)
print("L2 n_zero", int((np.abs(log_l2.coef_[0]) < 1e-12).sum()))
print("L2 acc", log_l2.score(x_test, y_test))
print("L2 auc", roc_auc_score(y_test, log_l2.predict_proba(x_test)[:, 1]))

### CLV-only card

CLV carries a large share of the ranking power; the other columns add a little desk context, not a new sport.

In [ ]:
Xc = df[["clv_cents"]].astype(float)
xc_tr, xc_te, yc_tr, yc_te = train_test_split(Xc, y, random_state=1, test_size=0.2)
lr_c = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr_c.fit(xc_tr, yc_tr)
print("CLV-only acc", lr_c.score(xc_te, yc_te))
print("CLV-only AUC", roc_auc_score(yc_te, lr_c.predict_proba(xc_te)[:, 1]))
print("full-card AUC was", roc_auc)

### Threshold as hold policy

In [ ]:
def predict_at(proba, t=0.5):
    return (proba >= t).astype(int)

p1 = y_pred_prob[:, 1]
print(f"{'t':>6} {'prec':>8} {'rec':>8} {'FP':>6} {'FN':>6} {'acc':>8}")
for t in (0.30, 0.40, 0.50, 0.60):
    pred = predict_at(p1, t)
    cm = confusion_matrix(y_test, pred)
    print(f"{t:6.2f} {precision_score(y_test, pred, zero_division=0):8.3f} "
          f"{recall_score(y_test, pred, zero_division=0):8.3f} "
          f"{cm[0,1]:6d} {cm[1,0]:6d} {accuracy_score(y_test, pred):8.3f}")

## 7. More practice

### Injury / weather / live / favorite

`injury_star` is the largest new coefficient (≈ −0.50). AUC edges up to about 0.64.

In [ ]:
cols2 = feature_cols + ["injury_star", "weather_flag", "live_bet", "is_favorite"]
X2 = pd.get_dummies(df[cols2], drop_first=True).astype(float)
x2_tr, x2_te, y2_tr, y2_te = train_test_split(X2, y, random_state=1, test_size=0.2)
lr2 = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr2.fit(x2_tr, y2_tr)
print(pd.DataFrame({"var": x2_tr.columns, "coef": lr2.coef_[0]})
      .reindex(pd.Series(np.abs(lr2.coef_[0]), index=x2_tr.columns)
               .sort_values(ascending=False).index)
      .head(8))
print("expanded AUC", roc_auc_score(y2_te, lr2.predict_proba(x2_te)[:, 1]))
print("expanded acc", lr2.score(x2_te, y2_te))

### Balanced weights — more tickets flagged Won

In [ ]:
log_bal = LogisticRegression(
    C=0.05, penalty="l1", solver="liblinear", class_weight="balanced"
)
log_bal.fit(x_train, y_train)
pred_b = log_bal.predict(x_test)
print("balanced acc", accuracy_score(y_test, pred_b))
print("balanced recall", recall_score(y_test, pred_b))
print("balanced prec", precision_score(y_test, pred_b))
print("balanced auc", roc_auc_score(y_test, log_bal.predict_proba(x_test)[:, 1]))
print(confusion_matrix(y_test, pred_b))

### Sport slice

In [ ]:
# sport dummies on the test fold; MLB is the dropped reference
nfl = x_test["sport_NFL"] == 1
nba = x_test["sport_NBA"] == 1
other = ~(nfl | nba)
pred = y_pred
print("recall NFL ", recall_score(y_test[nfl], pred[nfl]) if nfl.any() else None)
print("recall NBA ", recall_score(y_test[nba], pred[nba]) if nba.any() else None)
print("recall rest", recall_score(y_test[other], pred[other]))
print("base NFL ", y_test[nfl].mean() if nfl.any() else None)
print("base NBA ", y_test[nba].mean() if nba.any() else None)
print("base rest", y_test[other].mean())

### Which metric when?

In [ ]:
limit_desk = (
    "Precision (and a high threshold): limiting the wrong customer is cheap; "
    "missing a sharp who is beating the close is expensive."
)
hot_pick = (
    "Do not ship this. A 0.21 recall model is not a content engine; "
    "push notifications on weak scores create harm and chargebacks."
)
board_pack = (
    "Accuracy 0.627 next to the 0.611 Lost baseline, plus AUC 0.63 — "
    "the lift is real and small. Do not show accuracy alone."
)
print(limit_desk); print(hot_pick); print(board_pack)

## 8. Simulation

In [ ]:
# --- editable parameters ---
C = 0.05
N = 6000
N_REPS = 12
NOISE = 0.00
T = 0.50
SEED = 1
# ---------------------------
rng = np.random.default_rng(SEED)
rows = []
for r in range(N_REPS):
    idx = rng.integers(0, len(X), size=N)
    Xs = X.iloc[idx].reset_index(drop=True)
    ys = y[idx].copy()
    xtr, xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=SEED + r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy()
        ytr[flip] = 1 - ytr[flip]
    m = LogisticRegression(C=C, penalty="l1", solver="liblinear", max_iter=2000)
    m.fit(xtr, ytr)
    p = m.predict_proba(xte)[:, 1]
    pred = (p >= T).astype(int)
    rows.append({
        "acc": accuracy_score(yte, pred),
        "recall": recall_score(yte, pred, zero_division=0),
        "prec": precision_score(yte, pred, zero_division=0),
        "auc": roc_auc_score(yte, p),
        "n_zero": int((np.abs(m.coef_[0]) < 1e-12).sum()),
    })
sim = pd.DataFrame(rows)
print(sim.describe().round(3))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col in zip(axes, ["acc", "recall", "auc"]):
    ax.hist(sim[col], bins=8, color="#4c78a8", edgecolor="white")
    ax.set_title(col)
plt.suptitle(f"Bet_LogReg simulation  C={C}  N={N}  noise={NOISE}  T={T}")
plt.tight_layout()
plt.show()

## 9. Audience rewrite

In [ ]:
expert = (
    "L1-logistic (C=0.05, liblinear) on 12 dummy/continuous pre-game columns "
    "yields test AUC 0.633 and accuracy 0.627 versus a 0.611 Lost baseline. "
    "At t=0.5 Won-recall is 0.21 (TP=200, FN=749). Signs match market microstructure: "
    "CLV and line_move raise log-odds, public_pct lowers them. Injury on the expanded "
    "card is ≈ −0.50. This is a weak ranking screen. It does not identify +EV after juice, "
    "and CLV is only known after the close — leaking it into a pre-game poster is leakage."
)
technician = (
    "Read data/sports_tickets.csv. Dummy-encode clv_cents, public_pct, line_move, "
    "hours_to_start, stake_usd, is_home, sport, market (drop_first, astype float). "
    "Fit LogisticRegression(C=0.05, penalty='l1', solver='liblinear') on an 80/20 seed=1 split. "
    "Save intercept, non-zero coef table, CM [[1305,146],[749,200]], AUC. "
    "Production variant is the StandardScaler pipeline plus a configurable hold threshold."
)
executive = (
    "A simple ticket model is only a couple of points better than always expecting the "
    "customer to lose, which is already what juice is for. It is decent at ranking "
    "(AUC about 0.63) and very conservative at the default cutoff — it flags about "
    "one in five actual cashes. Use it to sort tickets for review, not to promise picks. "
    "CLV and fading public money are the levers. This file is synthetic."
)
nonspecialist = (
    "We asked a yes/no question: did this bet cash? Most tickets in the file lost, "
    "which is normal once the sportsbook takes its cut. The model looks at whether "
    "the number moved, how many people bet that side, and whether the ticket beat "
    "the last number. It is a bit better than a coin flip at ranking tickets and "
    "very cautious about saying 'this one wins.' It is not a lock, not a tip, and "
    "not a reason to stake more."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. Good fit vs not a fit

In [ ]:
good_fit = [
    "1. Binary ticket / cover / over-under labels with one row per wager.",
    "2. Desk wants a sparse, readable hold-list scorecard (L1) not a black-box bot.",
    "3. Teaching juice, class imbalance, and why accuracy lies in a book.",
    "4. Baseline before gradient-boosted models on the same blotter.",
    "5. Threshold workshops: limit / shade / no-action as a cutoff, not a vibe.",
    "6. CLV diagnostics — does beating the close actually show up in cash rate?",
    "7. Public-fade studies with an explicit dummy for side popularity.",
    "8. Injury / weather / live-bet ablation for trading ops.",
    "9. Monte-Carlo on C, n, and mis-keyed results before locking a spec.",
    "10. Responsible-gambling education: show how weak a 'model' still is after juice.",
]
not_a_fit = [
    "1. Live betting advice or social-media picks. AUC 0.63 does not pay −110.",
    "2. Using closing-line value as if it were known at ticket time (leakage).",
    "3. Causal claims ('the public causes losses'). This is a priced market.",
    "4. Problem-gambling targeting ('find losers and keep them betting').",
    "5. Bankroll / unit-size decisions. Logistic P(win) is not a Kelly input until calibrated and juiced.",
]
for row in good_fit: print(row)
print("--- not a fit ---")
for row in not_a_fit: print(row)

## 11. Done checklist

- [x] 12,000-row synthetic blotter
- [x] 61/39 imbalance vs Lost baseline 0.611
- [x] 12-column dummy matrix
- [x] L1 model, CM [[1305,146],[749,200]], acc ≈ 0.627, AUC ≈ 0.633
- [x] Sparse coefs: CLV +, public % −, line move +
- [x] Alternates, injury card, balanced weights, sport slice, simulation
- [x] Four-audience rewrite + good-fit list + no-advice disclaimer